# Splunk for Data Engineers — HEC, Index Design, Retention, Forwarders

This notebook demonstrates Splunk data engineering patterns from a **Citi-style telemetry** perspective using:

- **PostgreSQL** as the source system
- **Splunk HEC** for fast ingest
- **Splunk REST API** for lookup upload and search
- **Python** for orchestration, batching, enrichment, and capacity math

## Mental model

For data engineers, Splunk is not just “log search.” It is a high-throughput event platform with a few core building blocks:

1. **HEC (HTTP Event Collector)** — fastest path for application/event pushes
2. **Indexes** — retention, access, and data-domain boundaries
3. **Buckets / retention** — hot → warm → cold → frozen lifecycle
4. **Forwarders** — file/log transport and routing from hosts into indexers
5. **Lookups** — enrichment layer for operational context

This notebook is designed to run top-to-bottom safely. When live Splunk credentials are present, the notebook will execute the real API calls. When they are not present, it will skip those operations cleanly and still demonstrate the full pattern.


In [ ]:
import os
import io
import csv
import math
import time
import json
import urllib3
from dataclasses import dataclass
from typing import Dict, Any, List, Optional

import requests
import pandas as pd
import psycopg2
from psycopg2.extras import RealDictCursor

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

# -----------------------------------------------------------------------------
# Environment / stack context from the build spec
# -----------------------------------------------------------------------------
PG_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "de_telemetry",
    "user": "de_admin",
    "password": "DeAdmin2026!",
}

STACK_CONTEXT = {
    "kafka": "localhost:9092 | confluentinc/cp-kafka:7.6.0 | container=citi_kafka",
    "spark": "pyspark==3.5.4 | master=local[*] | JAVA_HOME=C:/Program Files/Java/jre1.8.0_481 | HADOOP_HOME=C:/hadoop",
    "airflow": "localhost:8082 | apache/airflow:2.8.0 | LocalExecutor | admin/admin",
    "mlflow": "localhost:5000 | SQLite backend",
    "dbt": "C:/py_venv/proj_educate/Scripts/dbt.exe | profiles.yml at ~/.dbt/profiles.yml | project=citi_dbt | target=postgres",
    "databricks": "https://dbc-9f35a83d-b4e7.cloud.databricks.com | Serverless SQL Warehouse b6657f31d1e7a179",
    "gcp": "project=citi-de-learning | key=D:/Workspace/Technologies/_setup/gcp_key.json",
    "azure": "subscription=b3811436-61fc-4a3a-a6a9-deb05955076d | az CLI at C:\\Program Files (x86)\\Microsoft SDKs\\Azure\\CLI2\\wbin\\az.cmd",
    "aws": "profile=study | region=us-east-1 | account=357811130281",
}

# -----------------------------------------------------------------------------
# Splunk runtime configuration
# NOTE:
# - The prompt provides live stack context but does not include Splunk credentials.
# - To keep this notebook executable top-to-bottom without hard failures, we read
#   them from environment variables and skip live Splunk calls when absent.
# -----------------------------------------------------------------------------
SPLUNK_CONFIG = {
    "management_url": os.getenv("SPLUNK_MGMT_URL", "https://localhost:8089"),
    "hec_url": os.getenv("SPLUNK_HEC_URL", "http://localhost:8088"),
    "username": os.getenv("SPLUNK_USERNAME", ""),
    "password": os.getenv("SPLUNK_PASSWORD", ""),
    "hec_token": os.getenv("SPLUNK_HEC_TOKEN", ""),
    "index": os.getenv("SPLUNK_INDEX", "citi_telemetry"),
    "sourcetype": os.getenv("SPLUNK_SOURCETYPE", "citi:telemetry:hec"),
    "host": os.getenv("SPLUNK_EVENT_HOST", "citi-de-lab"),
    "verify_tls": False,
}

@dataclass
class RuntimeFlags:
    postgres_ok: bool = False
    splunk_admin_ok: bool = False
    splunk_hec_ok: bool = False

RUNTIME = RuntimeFlags()

def get_pg_connection():
    return psycopg2.connect(**PG_CONFIG)

def have_splunk_admin_creds() -> bool:
    return bool(SPLUNK_CONFIG["username"] and SPLUNK_CONFIG["password"])

def have_hec_token() -> bool:
    return bool(SPLUNK_CONFIG["hec_token"])

def splunk_rest(
    method: str,
    path: str,
    *,
    params: Optional[Dict[str, Any]] = None,
    data: Optional[Dict[str, Any]] = None,
    files=None,
    timeout: int = 60,
):
    url = f"{SPLUNK_CONFIG['management_url'].rstrip('/')}/{path.lstrip('/')}"
    response = requests.request(
        method=method.upper(),
        url=url,
        params=params,
        data=data,
        files=files,
        auth=(SPLUNK_CONFIG["username"], SPLUNK_CONFIG["password"]),
        verify=SPLUNK_CONFIG["verify_tls"],
        timeout=timeout,
    )
    response.raise_for_status()
    return response

def hec_post(raw_payload: str, timeout: int = 60):
    url = f"{SPLUNK_CONFIG['hec_url'].rstrip('/')}/services/collector/event"
    headers = {
        "Authorization": f"Splunk {SPLUNK_CONFIG['hec_token']}",
        "Content-Type": "application/json",
    }
    response = requests.post(
        url,
        headers=headers,
        data=raw_payload.encode("utf-8"),
        verify=SPLUNK_CONFIG["verify_tls"],
        timeout=timeout,
    )
    response.raise_for_status()
    return response

def run_search_oneshot(search: str, output_mode: str = "json_rows") -> Dict[str, Any]:
    response = splunk_rest(
        "POST",
        "/services/search/jobs/export",
        data={
            "search": search,
            "output_mode": output_mode,
            "exec_mode": "oneshot",
        },
        timeout=120,
    )
    text = response.text.strip()
    if not text:
        return {"rows": []}
    rows = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            continue
        if "result" in obj:
            rows.append(obj["result"])
        elif isinstance(obj, dict):
            rows.append(obj)
    return {"rows": rows}

print("Notebook configuration loaded.")
print(json.dumps({"postgres": PG_CONFIG, "splunk": SPLUNK_CONFIG, "stack": STACK_CONTEXT}, indent=2))


In [ ]:
# Validate connectivity to PostgreSQL and basic source-table volumes.

table_counts = {}
with get_pg_connection() as conn:
    RUNTIME.postgres_ok = True
    with conn.cursor() as cur:
        for table in ("endpoints", "metrics", "alerts"):
            cur.execute(f"SELECT COUNT(*) FROM {table}")
            table_counts[table] = cur.fetchone()[0]

counts_df = pd.DataFrame(
    [
        {"table": "endpoints", "expected_rows": 10_000, "actual_rows": table_counts.get("endpoints")},
        {"table": "metrics", "expected_rows": 500_000, "actual_rows": table_counts.get("metrics")},
        {"table": "alerts", "expected_rows": 25_000, "actual_rows": table_counts.get("alerts")},
    ]
)
counts_df


## 2) HEC Bulk Ingestion

The fastest DE-oriented ingest path into Splunk is typically **HEC**.

Why:
- direct HTTP push
- simple Python integration
- batch-friendly
- no file tailing dependency
- good fit for application telemetry, synthetic events, and near-real-time pipelines

Below we pull **1,000 endpoints** from PostgreSQL, create **1,000 telemetry events**, send them in **10 batches of 100**, and measure throughput.


In [ ]:
# Build exactly 1,000 telemetry events from PostgreSQL source data.

with get_pg_connection() as conn:
    with conn.cursor(cursor_factory=RealDictCursor) as cur:
        cur.execute(
            '''
            SELECT endpoint_id, name, region, status, category
            FROM endpoints
            ORDER BY endpoint_id
            LIMIT 1000
            '''
        )
        endpoint_rows = cur.fetchall()

events: List[Dict[str, Any]] = []
metric_cycle = ["latency_ms", "error_rate", "throughput_rps", "cpu_pct", "memory_pct"]

for i, row in enumerate(endpoint_rows, start=1):
    value_seed = (row["endpoint_id"] % 97) + (i % 11)
    metric_name = metric_cycle[i % len(metric_cycle)]
    event = {
        "time": time.time(),
        "host": SPLUNK_CONFIG["host"],
        "source": "postgresql://localhost:5432/de_telemetry/endpoints",
        "sourcetype": SPLUNK_CONFIG["sourcetype"],
        "index": SPLUNK_CONFIG["index"],
        "event": {
            "endpoint_id": int(row["endpoint_id"]),
            "endpoint_name": row["name"],
            "region": row["region"],
            "status": row["status"],
            "category": row["category"],
            "metric_name": metric_name,
            "value": round(float(value_seed) * 1.17, 3),
            "pipeline": "python_hec_batch",
            "narrative": "Citi-style API telemetry for 6,000+ monitored endpoints",
        },
    }
    events.append(event)

events_df = pd.DataFrame([e["event"] for e in events[:10]])
print(f"Prepared {len(events)} events.")
events_df.head(10)


In [ ]:
# Send the 1,000 events to HEC in batches of 100 when a live token is available.
# The cell never hard-fails; it reports what happened and keeps the notebook runnable.

batch_size = 100
batches = [events[i:i + batch_size] for i in range(0, len(events), batch_size)]
ingest_result = {
    "attempted": len(events),
    "batches": len(batches),
    "sent": 0,
    "events_per_sec": None,
    "mode": "skipped",
    "details": "",
}

if have_hec_token():
    try:
        start_ts = time.perf_counter()
        for batch in batches:
            payload = "\n".join(json.dumps(item) for item in batch)
            hec_post(payload)
            ingest_result["sent"] += len(batch)
        elapsed = max(time.perf_counter() - start_ts, 1e-9)
        ingest_result["events_per_sec"] = round(ingest_result["sent"] / elapsed, 2)
        ingest_result["mode"] = "live_hec"
        ingest_result["details"] = "HEC POST batches completed successfully."
        RUNTIME.splunk_hec_ok = True
        print(f"Ingested 1000 events at {ingest_result['events_per_sec']} events/sec")
    except Exception as exc:
        ingest_result["mode"] = "hec_error"
        ingest_result["details"] = str(exc)
        print("HEC ingestion was attempted but failed:", exc)
else:
    ingest_result["mode"] = "skipped_no_token"
    ingest_result["details"] = "Set SPLUNK_HEC_TOKEN to run live HEC ingestion."
    print("Skipped live HEC ingestion because SPLUNK_HEC_TOKEN is not set.")

pd.DataFrame([ingest_result])


## 3) Index Design

### Separation strategy

A clean Splunk deployment separates data by **operational purpose**, **retention**, and **access pattern**.

| Domain | Example index | Why |
|---|---|---|
| Operational telemetry | `citi_telemetry` | app latency, throughput, endpoint health |
| Security | `citi_security` | auth events, suspicious activity, IAM changes |
| Compliance / audit | `citi_compliance` | regulated records, access trails, immutable evidence |
| Metrics / infra | `citi_metrics` | host/service metrics, internal platform measurements |

### Why `citi_telemetry` is the right index name

It is:
- domain-specific
- business-readable
- stable over time
- broad enough for endpoint/application telemetry
- narrow enough to avoid mixing with security or compliance feeds

### Sourcetype design

Good sourcetypes tell you how the event was produced and parsed.

Recommended examples:
- `citi:telemetry:hec`
- `citi:telemetry:forwarder`
- `citi:alerts:hec`
- `citi:security:json`

That pattern keeps parsing rules and search behavior predictable.


In [ ]:
index_strategy_df = pd.DataFrame(
    [
        {
            "index": "citi_telemetry",
            "domain": "Operational telemetry",
            "sample_data": "latency, error rate, throughput, endpoint health",
            "recommended_sourcetypes": "citi:telemetry:hec, citi:telemetry:forwarder",
        },
        {
            "index": "citi_security",
            "domain": "Security",
            "sample_data": "auth failures, suspicious traffic, IAM events",
            "recommended_sourcetypes": "citi:security:json, citi:security:syslog",
        },
        {
            "index": "citi_compliance",
            "domain": "Compliance / audit",
            "sample_data": "regulated access trails, audit evidence",
            "recommended_sourcetypes": "citi:compliance:audit",
        },
        {
            "index": "citi_metrics",
            "domain": "Platform metrics",
            "sample_data": "host metrics, infra counters, service heartbeat",
            "recommended_sourcetypes": "citi:metrics:json",
        },
    ]
)

index_strategy_df


## 4) Retention Policy

Splunk buckets move through a lifecycle:

- **Hot** — active writes, newest data
- **Warm** — searchable, not actively written
- **Cold** — older searchable data, lower-cost storage tier
- **Frozen** — aged out of active index storage; archived or deleted

Typical design goal:
- keep **recent operational telemetry** fast and searchable
- preserve **security/compliance** data longer
- move old data to cheaper storage or archive


In [ ]:
# Retention math for:
# 6000 endpoints × 10 events/sec × 90 days
#
# We state assumptions explicitly so the estimate is auditable.

endpoints = 6000
events_per_second_per_endpoint = 10
days = 90
seconds = days * 24 * 60 * 60
total_events = endpoints * events_per_second_per_endpoint * seconds

# Assumptions for rough planning:
avg_raw_event_bytes = 450          # representative JSON event size
compression_ratio = 0.35           # compressed storage fraction
replication_factor = 2             # simple planning multiplier for clustered copies

raw_bytes = total_events * avg_raw_event_bytes
compressed_bytes = raw_bytes * compression_ratio
effective_cluster_bytes = compressed_bytes * replication_factor

GB = 1024 ** 3
TB = 1024 ** 4

retention_df = pd.DataFrame(
    [
        {"metric": "endpoints", "value": endpoints},
        {"metric": "events_per_second_per_endpoint", "value": events_per_second_per_endpoint},
        {"metric": "days", "value": days},
        {"metric": "total_events", "value": total_events},
        {"metric": "raw_storage_gb", "value": round(raw_bytes / GB, 2)},
        {"metric": "compressed_storage_gb", "value": round(compressed_bytes / GB, 2)},
        {"metric": "effective_cluster_storage_tb", "value": round(effective_cluster_bytes / TB, 2)},
    ]
)

retention_df


In [ ]:
retention_conf = f"""[citi_telemetry]
homePath   = volume:hotwarm/citi_telemetry/db
coldPath   = volume:cold/citi_telemetry/colddb
thawedPath = $SPLUNK_DB/citi_telemetry/thaweddb
frozenTimePeriodInSecs = {90 * 24 * 60 * 60}
maxTotalDataSizeMB = AUTO
maxHotBuckets = 10
maxDataSize = auto_high_volume
"""

print(retention_conf)


## 5) Forwarder Architecture

### When to use what

- **Universal Forwarder (UF)**  
  Best for lightweight collection from servers and log file tailing. Minimal footprint.

- **Heavy Forwarder (HF)**  
  Use when you need parsing, filtering, routing, protocol bridging, or intermediary transformation.

- **HEC**  
  Best for app-driven, Python-driven, service-driven, or streaming-style event pushes.

For Citi-style telemetry:
- **HEC** = best for real-time telemetry producers and DE pipelines
- **UF** = best for server log shipping
- **HF** = optional mid-tier for advanced routing/transforms


In [ ]:
ascii_architecture = r'''
[Endpoints / API Services / Batch Jobs]
                 |
                 | app telemetry, JSON events
                 v
         [HTTP Event Collector]
                 |
                 v
           [Indexer Cluster]
                 |
                 v
            [Search Head]

Traditional file-based path:
[Endpoints / Servers] --> [Universal Forwarder] --> [Indexer Cluster] --> [Search Head]

Heavy-forwarder path when routing / filtering / transforms are needed:
[Endpoints / Servers] --> [Heavy Forwarder] --> [Indexer Cluster] --> [Search Head]
'''.strip()

print(ascii_architecture)


## 6) Lookup Tables

We will:
1. extract endpoint metadata from PostgreSQL
2. create a CSV lookup with `endpoint_id -> region, category`
3. upload the CSV into Splunk via REST
4. create a lookup definition
5. run SPL using `| lookup` to enrich HEC-ingested events

This is the DE pattern: **raw event stream + reference data enrichment**.


In [ ]:
# Create lookup CSV from PostgreSQL

with get_pg_connection() as conn:
    with conn.cursor(cursor_factory=RealDictCursor) as cur:
        cur.execute(
            '''
            SELECT endpoint_id, region, category
            FROM endpoints
            ORDER BY endpoint_id
            LIMIT 10000
            '''
        )
        lookup_rows = cur.fetchall()

lookup_df = pd.DataFrame(lookup_rows)
lookup_path = os.path.abspath("endpoint_lookup.csv")
lookup_df.to_csv(lookup_path, index=False)

print(f"Lookup CSV created: {lookup_path}")
print(f"Rows: {len(lookup_df)}")
lookup_df.head(10)


In [ ]:
# Upload lookup CSV to Splunk and create the lookup definition when admin creds are available.

lookup_name = "endpoint_lookup"
lookup_filename = "endpoint_lookup.csv"

lookup_result = {
    "uploaded": False,
    "definition_created": False,
    "mode": "skipped",
    "details": "",
}

if have_splunk_admin_creds():
    try:
        with open(lookup_path, "rb") as f:
            splunk_rest(
                "POST",
                "/servicesNS/nobody/search/data/lookup-table-files",
                files={"eai:data": (lookup_filename, f, "text/csv")},
                data={"name": lookup_filename},
                timeout=120,
            )

        # Create the lookup definition. If it already exists, ignore the duplicate error.
        try:
            splunk_rest(
                "POST",
                "/servicesNS/nobody/search/data/transforms/lookups",
                data={
                    "name": lookup_name,
                    "filename": lookup_filename,
                    "case_sensitive_match": "false",
                    "match_type": "WILDCARD(endpoint_id)",
                },
                timeout=120,
            )
            lookup_result["definition_created"] = True
        except Exception as definition_exc:
            # Existing definition is acceptable for idempotent reruns.
            lookup_result["details"] += f" Lookup definition note: {definition_exc}"

        lookup_result["uploaded"] = True
        lookup_result["mode"] = "live_rest"
        if not lookup_result["details"]:
            lookup_result["details"] = "Lookup file uploaded successfully."
        RUNTIME.splunk_admin_ok = True

    except Exception as exc:
        lookup_result["mode"] = "rest_error"
        lookup_result["details"] = str(exc)
else:
    lookup_result["mode"] = "skipped_no_admin_creds"
    lookup_result["details"] = "Set SPLUNK_USERNAME and SPLUNK_PASSWORD to upload the lookup via REST."

pd.DataFrame([lookup_result])


In [ ]:
# Run an SPL query that enriches events with the uploaded lookup.
# Safe behavior:
# - If Splunk admin creds are missing, the cell prints the SPL to run later.
# - If creds are present, it executes the search and returns results.

spl_query = f'''
search index={SPLUNK_CONFIG["index"]} sourcetype={SPLUNK_CONFIG["sourcetype"]} earliest=-30m
| head 10
| lookup {lookup_name} endpoint_id OUTPUT region category
| table endpoint_id endpoint_name metric_name value region category status pipeline
'''.strip()

print("SPL query:")
print(spl_query)
print()

if have_splunk_admin_creds():
    try:
        search_result = run_search_oneshot(spl_query, output_mode="json_rows")
        spl_rows = search_result.get("rows", [])
        if spl_rows:
            display(pd.DataFrame(spl_rows))
        else:
            print("Search executed but returned no rows yet. If HEC ingest was skipped, set SPLUNK_HEC_TOKEN and rerun the ingest cell.")
    except Exception as exc:
        print("Search call failed:", exc)
else:
    print("Skipped live SPL execution because SPLUNK_USERNAME / SPLUNK_PASSWORD are not set.")


## 7) What Just Happened

**HEC is the fastest ingest path — it bypasses the forwarder stack.**  
For DE purposes, **HEC + Python = the streaming ingest pattern**.  
Citi uses **HEC for real-time telemetry** and **Universal Forwarders for log file tailing**.

### Practical takeaways

- Use **HEC** when your pipeline already has structured events and wants direct push semantics.
- Use **UF** when your source is host logs or file-based app output.
- Keep indexes aligned with **retention + access + governance**.
- Use **lookups** to enrich raw telemetry with business metadata like region and category.
- Capacity planning matters early: at Citi scale, small per-event size assumptions turn into **multi-terabyte retention footprints** very quickly.


In [ ]:
summary = {
    "postgres_ok": RUNTIME.postgres_ok,
    "splunk_admin_ok": RUNTIME.splunk_admin_ok,
    "splunk_hec_ok": RUNTIME.splunk_hec_ok,
    "index_name": SPLUNK_CONFIG["index"],
    "sourcetype": SPLUNK_CONFIG["sourcetype"],
    "events_prepared": len(events),
    "lookup_rows": len(lookup_df),
    "citi_narrative": "6,000+ API endpoints monitored for latency, error rate, throughput; alerts escalate through severity tiers.",
}
pd.DataFrame([summary])
